In [ ]:
import os, sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from nbafuns import *

from nba_api.stats.endpoints import leaguedashplayerstats
data_DIR = "../data/salary/"

"star" = made 1+ all-star game   
"starter" = 40+ games started in a season  
"bench" = 20+ mpg in 40+ games in a season  
"got some playing time" = 40+ games total  
"cup of coffee or less" = <40  

In [ ]:
seasons = range(2015,2025)
sea = []
sal = []
for season in seasons:
    df1 = pd.read_parquet(data_DIR+f"NBA_Spotrac_{season}.parquet")
    df1.columns = map(str.lower,df1.columns)
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        measure_type_detailed_defense="Base",
        per_mode_detailed="PerGame",
        season=get_ss(season),
        season_type_all_star="Regular Season",
        # starter_bench_nullable="Starters"
    )
    df2_1 = stats.get_data_frames()[0]
    time.sleep(1)
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        measure_type_detailed_defense="Base",
        per_mode_detailed="PerGame",
        season=get_ss(season),
        season_type_all_star="Regular Season",
        starter_bench_nullable="Starters"
    )
    df2_2 = stats.get_data_frames()[0]
    time.sleep(1)
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        measure_type_detailed_defense="Base",
        per_mode_detailed="PerGame",
        season=get_ss(season),
        season_type_all_star="Regular Season",
        starter_bench_nullable="Bench"
    )
    df2_3 = stats.get_data_frames()[0]
    df3_1 = df2_1[["PLAYER_NAME","TEAM_ABBREVIATION","PLAYER_ID","GP","MIN"]]
    df3_2 = df2_2[["PLAYER_ID","GP","MIN"]]
    df3_3 = df2_3[["PLAYER_ID","GP","MIN"]]
    df3_4 = pd.merge(df3_1,df3_2,on="PLAYER_ID",suffixes=["","_x"],how="left")
    df3 = pd.merge(df3_4,df3_3,on="PLAYER_ID",suffixes=["","_y"],how="left")
    df3.columns = map(str.lower,df3.columns)
    df3 = df3.fillna(0)
    df3["gp_s"] = round(df3["gp_x"]/df3["gp"],2)
    df3 = df3.drop(columns=["min_x","min_y"])
    df3['player_name'] = (
        df3['player_name']
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
    )
    df3["starter"] = df3["gp_s"] >=0.5
    df4 = df3[df3["starter"]]
    df4 = df4.query("gp>=10 & min>=20").reset_index(drop=True)
    df5 = pd.merge(df4,df1,left_on="player_name",right_on="player",how="left")
    df5 = df5.sort_values("salary",ascending=False).reset_index(drop=True)
    df5 = df5[['player_name', 'team_abbreviation', 'player_id', 'gp', 'min', 'salary','season']]
    sal.append(round(df5['salary'].mean()/1_000_000,2))
    sea.append(season+1)
    print(f"{season+1}: {round(df5['salary'].mean())}")

In [ ]:
dfs = pd.read_parquet(data_DIR + "NBA_Salary_Cap.parquet")
dfs = dfs[["season","salary_cap"]]
dfs = dfs.rename(columns={"season":"Season"})
dfs["salary_cap"] = dfs["salary_cap"]/1e6

In [ ]:
dfp = pd.DataFrame({"Season":sea,"Salary":sal})
dfp = pd.merge(dfp,dfs,on="Season")
dfp["Salary_pct"] = dfp["Salary"]/dfp["salary_cap"]

In [ ]:
p = (
    ggplot(dfp)
    + aes(x="Season",y="Salary",group=0)
    + geom_line()
    + geom_text(aes(label="Salary"),nudge_y=0.5)
    + geom_point(size=2)
    + theme_idv
    + theme(
        plot_subtitle=element_text(size=14)
    )
    + labs(
        title="NBA Average Starter Salary",
        subtitle="Starter: 20+MPG, >50% Games Started, Min 10 Games Played",
        caption="@sradjoker | source: spotrac, nba.com/stats",
        y="Salary in Million $"
    )
)
p

In [ ]:
p = (
    ggplot(dfp)
    + aes(x="Season",y="Salary_pct",group=0)
    + geom_line()
    # + geom_text(aes(label="Salary"),nudge_y=0.5)
    + geom_point(size=2)
    + scale_y_continuous(labels=percent_format())
    + scale_x_continuous(breaks=[2016,2020,2025])

    + theme_idv
    + theme(
        plot_subtitle=element_text(size=12)
    )
    + labs(
        title="NBA Average Starter Salary: % of Cap",
        subtitle="Starter: 20+MPG, >50% Games Started, Min 10 Games Played",
        caption="@sradjoker | source: spotrac, nba.com/stats",
        y="Salary as % of Cap"
    )
)
p